# English Tutor: fine-tune Qwen3-4B-Instruct on a free Colab GPU

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**. Near the top it asks you to allow Google Drive and to upload `train.jsonl` and `val.jsonl` (from your `data/` folder). After that it runs on its own (~30–40 min).
3. When it finishes, download `english-tutor.Q4_K_M.gguf` from `MyDrive/english-tutor/` and put it next to your `Modelfile`.

In [ ]:
!pip install -q unsloth

In [ ]:
# Google Drive (results are saved here) + your data
import os
from google.colab import drive, files

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/english-tutor"
os.makedirs(OUT, exist_ok=True)

files.upload()  # pick train.jsonl and val.jsonl

In [ ]:
# Load the base model in 4-bit and attach a small trainable LoRA adapter (QLoRA)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=1024,  # tutor examples are short
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Turn each conversation into the model's chat format
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})
SYSTEM_PROMPT = ds["train"][0]["messages"][0]["content"]  # reused for testing below
ds = ds.map(lambda x: {"text": tokenizer.apply_chat_template(x["messages"], tokenize=False)},
            remove_columns=["messages"])
print(ds)
print(ds["train"][0]["text"])

In [ ]:
# Train. Loss is computed only on the tutor's replies, not on the prompt or the student's sentence.
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        weight_decay=0.01,
        optim="adamw_8bit",
        logging_steps=10,
        eval_strategy="epoch",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

In [ ]:
# Quick check on sentences it has never seen
FastLanguageModel.for_inference(model)

for sentence in ["He don't knows the answer.", "I visited my grandparents last summer.", "She is married with a doctor since two years."]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": sentence}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    print(">>", sentence)
    print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True), "\n")

In [ ]:
# Save: LoRA adapter (small backup) first, then one GGUF file for Ollama (~10-15 min)
import glob, shutil

model.save_pretrained("lora")
tokenizer.save_pretrained("lora")
shutil.copytree("lora", f"{OUT}/lora", dirs_exist_ok=True)

model.save_pretrained_gguf("english-tutor", tokenizer, quantization_method="q4_k_m")
gguf = min(glob.glob("english-tutor*/**/*.gguf", recursive=True) + glob.glob("english-tutor*.gguf"),
           key=os.path.getsize)  # the smallest one is the 4-bit file
shutil.copy(gguf, f"{OUT}/english-tutor.Q4_K_M.gguf")
print(f"Done: {OUT}/english-tutor.Q4_K_M.gguf ({os.path.getsize(gguf) / 1e9:.1f} GB)")

In [ ]:
# Optional: upload the GGUF to Hugging Face so your Space can serve it (skip if running only on your laptop)
from huggingface_hub import HfApi, login

login()  # paste a token with "write" access from huggingface.co/settings/tokens
api = HfApi()
repo = api.create_repo("english-tutor-gguf", exist_ok=True).repo_id  # public, so Ollama can pull it
api.upload_file(path_or_fileobj=gguf, path_in_repo="english-tutor.Q4_K_M.gguf", repo_id=repo)
print(f"In your Space settings, add the variable MODEL_SOURCE = hf.co/{repo}:Q4_K_M")